# Lab: สร้าง RAG เองทีละขั้น ด้วย Ollama + Qwen3

เคส: ผู้ช่วยตอบคำถามแนวปฏิบัติของ **โรงพยาบาลสมมติเวช** (ข้อมูลทั้งหมดเป็นสิ่งสมมติ ใช้เพื่อการสอนเท่านั้น ไม่ใช่คำแนะนำทางการแพทย์)

เขียนด้วย Python ตรง ๆ ไม่ใช้ framework เพื่อให้เห็นกลไกทุกขั้น

| ขั้น | ทำอะไร |
|---|---|
| 0 | Setup และถามโมเดลโดยไม่มี RAG |
| 1 | Embedding และ similarity |
| 2 | โหลดเอกสาร ตัด chunk สร้าง index |
| 3 | Retrieve |
| 4 | Generate |
| 5 | ทดลองพัง |

**ก่อนเริ่ม:** เมนู Runtime → Change runtime type → เลือก **T4 GPU**

## ขั้น 0 · Setup

Cell นี้ติดตั้ง Ollama เปิด server และดาวน์โหลดโมเดล 2 ตัว (`qwen3:8b` สำหรับตอบ, `bge-m3` สำหรับ embedding) ใช้เวลาหลายนาที

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!sudo apt-get install -y -qq zstd          # ตัวติดตั้ง Ollama รุ่นใหม่ต้องใช้ zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install ollama faiss-cpu

In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

In [ ]:
!ollama pull qwen3:8b
!ollama pull bge-m3
!ollama list

### ชุดข้อมูล

Cell นี้สร้างเอกสารแนวปฏิบัติ 6 ไฟล์ของโรงพยาบาลสมมติเวชไว้ในโฟลเดอร์ `rag-med-dataset/guidelines/` ไม่ต้องอัปโหลดไฟล์ใด ๆ

| ไฟล์ | เนื้อหา |
|---|---|
| CPG-DM-2568.md | แนวปฏิบัติเบาหวาน ฉบับปัจจุบัน |
| CPG-DM-2565.md | แนวปฏิบัติเบาหวาน ฉบับเก่า (ยกเลิกแล้ว) |
| CPG-HT-2567.md | แนวปฏิบัติความดันโลหิตสูง |
| PROTOCOL-SEPSIS-2567.md | แนวปฏิบัติ sepsis |
| FORMULARY-2568.md | บัญชียา |
| MEMO-2568-014.md | ประกาศจำกัดการใช้ยาซูการ์ดิน |

**ทุกอย่างเป็นเรื่องสมมติ** ชื่อยาและเกณฑ์แต่งขึ้นเพื่อการสอน ห้ามใช้อ้างอิงทางคลินิก

In [ ]:
import os
os.makedirs("rag-med-dataset/guidelines", exist_ok=True)
DOCS = {}

DOCS["CPG-DM-2568.md"] = """# แนวปฏิบัติการดูแลผู้ป่วยเบาหวานชนิดที่ 2 โรงพยาบาลสมมติเวช ฉบับ พ.ศ. 2568

> เอกสารสมมติเพื่อการสอนเท่านั้น โรงพยาบาล ชื่อยา และเกณฑ์ทั้งหมดแต่งขึ้น ห้ามใช้อ้างอิงทางคลินิก

รหัสเอกสาร: CPG-DM-2568
สถานะ: ใช้งานปัจจุบัน มีผลตั้งแต่ 1 มกราคม 2568 และใช้แทนฉบับ CPG-DM-2565 ทั้งหมด

## 1. ขอบเขต
แนวปฏิบัตินี้ใช้กับผู้ป่วยผู้ใหญ่ที่ได้รับการวินิจฉัยเบาหวานชนิดที่ 2 ในคลินิกอายุรกรรมและคลินิกโรคเรื้อรังของโรงพยาบาลสมมติเวช ไม่ครอบคลุมหญิงตั้งครรภ์และผู้ป่วยเด็ก

## 2. ยาลำดับแรก
ยาลำดับแรกคือกลูโคนอร์ม (Gluconorm) รหัสยา GLN-850 ชนิดเม็ด 850 มิลลิกรัม ขนาดปกติสำหรับผู้ที่ไตทำงานปกติคือ 850 มิลลิกรัม วันละ 2 ครั้ง หลังอาหารเช้าและเย็น ให้เริ่มที่วันละ 1 ครั้งในสัปดาห์แรกเพื่อลดอาการคลื่นไส้ แล้วเพิ่มเป็นวันละ 2 ครั้งเมื่อผู้ป่วยทนยาได้

## 3. การปรับขนาดกลูโคนอร์มตามการทำงานของไต
ต้องมีผล eGFR ภายใน 3 เดือนก่อนสั่งยาทุกครั้ง และปรับขนาดตามตารางที่ 1

ตารางที่ 1 ขนาดกลูโคนอร์มตามค่า eGFR
| eGFR (มล./นาที/1.73 ตร.ม.) | ขนาดยา | การติดตาม |
|---|---|---|
| 60 ขึ้นไป | 850 มก. วันละ 2 ครั้ง | ตรวจ eGFR ปีละครั้ง |
| 45 ถึง 59 | 850 มก. วันละ 1 ครั้ง | ตรวจ eGFR ทุก 6 เดือน |
| 30 ถึง 44 | 425 มก. (ครึ่งเม็ด) วันละ 1 ครั้ง | ตรวจ eGFR ทุก 3 เดือน |
| ต่ำกว่า 30 | หยุดยา | ส่งปรึกษาอายุรแพทย์โรคไต |

ผู้ป่วยที่ eGFR ลดลงมากกว่า 10 หน่วยภายใน 3 เดือน ให้หยุดยาชั่วคราวและตรวจซ้ำภายใน 2 สัปดาห์ แม้ค่าจะยังสูงกว่า 30

## 4. ยาลำดับที่สอง
เมื่อใช้กลูโคนอร์มครบ 3 เดือนแล้ว HbA1c ยังสูงกว่าเป้าหมาย ให้เพิ่มยาไดอะเบกซ์ (Diabex-R) รหัสยา DBX-100 ขนาด 100 มิลลิกรัม วันละ 1 ครั้ง ห้ามเริ่มยาซูการ์ดิน (Sugardin) ในผู้ป่วยรายใหม่ ตามประกาศ MEMO-2568-014

## 5. เป้าหมายและการติดตาม
เป้าหมาย HbA1c ต่ำกว่าร้อยละ 7.0 ในผู้ป่วยทั่วไป และต่ำกว่าร้อยละ 8.0 ในผู้สูงอายุที่มีโรคร่วมหลายโรค ตรวจ HbA1c ทุก 3 เดือนจนถึงเป้าหมาย แล้วตรวจทุก 6 เดือน ตรวจตาและตรวจเท้าปีละครั้ง

## 6. การส่งต่อ
ส่งต่ออายุรแพทย์โรคต่อมไร้ท่อเมื่อ HbA1c สูงกว่าร้อยละ 10.0 สองครั้งติดกัน หรือเมื่อผู้ป่วยมีภาวะน้ำตาลต่ำรุนแรงมากกว่า 1 ครั้งใน 6 เดือน
"""

DOCS["CPG-DM-2565.md"] = """# แนวปฏิบัติการดูแลผู้ป่วยเบาหวานชนิดที่ 2 โรงพยาบาลสมมติเวช ฉบับ พ.ศ. 2565

> เอกสารสมมติเพื่อการสอนเท่านั้น โรงพยาบาล ชื่อยา และเกณฑ์ทั้งหมดแต่งขึ้น ห้ามใช้อ้างอิงทางคลินิก

รหัสเอกสาร: CPG-DM-2565
สถานะ: ยกเลิกแล้ว ถูกแทนที่ด้วยฉบับ CPG-DM-2568 ตั้งแต่ 1 มกราคม 2568 เก็บไว้เพื่อการอ้างอิงย้อนหลังเท่านั้น

## 1. ขอบเขต
แนวปฏิบัตินี้ใช้กับผู้ป่วยผู้ใหญ่ที่ได้รับการวินิจฉัยเบาหวานชนิดที่ 2 ในคลินิกอายุรกรรมของโรงพยาบาลสมมติเวช

## 2. ยาลำดับแรก
ยาลำดับแรกคือกลูโคนอร์ม (Gluconorm) รหัสยา GLN-850 ขนาด 850 มิลลิกรัม วันละ 2 ครั้ง หลังอาหาร

## 3. การปรับขนาดกลูโคนอร์มตามการทำงานของไต
ตรวจ eGFR อย่างน้อยปีละครั้ง และปรับขนาดตามตารางที่ 1

ตารางที่ 1 ขนาดกลูโคนอร์มตามค่า eGFR (ฉบับ 2565)
| eGFR (มล./นาที/1.73 ตร.ม.) | ขนาดยา |
|---|---|
| 45 ขึ้นไป | 850 มก. วันละ 2 ครั้ง |
| 30 ถึง 44 | 850 มก. วันละ 1 ครั้ง |
| ต่ำกว่า 30 | หยุดยา |

## 4. ยาลำดับที่สอง
เมื่อ HbA1c ยังสูงกว่าเป้าหมายหลังใช้กลูโคนอร์ม 3 เดือน ให้เพิ่มยาซูการ์ดิน (Sugardin) รหัสยา SGD-120 ขนาด 120 มิลลิกรัม วันละ 1 ครั้ง ก่อนอาหารเช้า

## 5. เป้าหมายและการติดตาม
เป้าหมาย HbA1c ต่ำกว่าร้อยละ 7.0 ตรวจ HbA1c ทุก 6 เดือน ตรวจตาปีละครั้ง
"""

DOCS["CPG-HT-2567.md"] = """# แนวปฏิบัติการดูแลผู้ป่วยความดันโลหิตสูง โรงพยาบาลสมมติเวช ฉบับ พ.ศ. 2567

> เอกสารสมมติเพื่อการสอนเท่านั้น โรงพยาบาล ชื่อยา และเกณฑ์ทั้งหมดแต่งขึ้น ห้ามใช้อ้างอิงทางคลินิก

รหัสเอกสาร: CPG-HT-2567
สถานะ: ใช้งานปัจจุบัน

## 1. การวินิจฉัย
วินิจฉัยความดันโลหิตสูงเมื่อค่าเฉลี่ยความดันที่วัดในคลินิกตั้งแต่ 140/90 มิลลิเมตรปรอทขึ้นไป จากการวัด 2 ครั้ง ห่างกันอย่างน้อย 1 สัปดาห์ หรือค่าเฉลี่ยความดันที่วัดที่บ้านตั้งแต่ 135/85 มิลลิเมตรปรอทขึ้นไป

## 2. การวัดความดันที่บ้าน
ให้ผู้ป่วยวัดความดันที่บ้านวันละ 2 ช่วง คือช่วงเช้าภายใน 1 ชั่วโมงหลังตื่นนอนและก่อนกินยา กับช่วงก่อนนอน ช่วงละ 2 ครั้ง ห่างกัน 1 นาที ติดต่อกัน 7 วัน ตัดค่าของวันแรกออก แล้วใช้ค่าเฉลี่ยของ 6 วันที่เหลือ ให้นั่งพัก 5 นาทีก่อนวัด และงดกาแฟกับบุหรี่ 30 นาทีก่อนวัด

## 3. ยาลำดับแรก
ยาลำดับแรกคือเพรสซิลอล (Pressilol) รหัสยา PSL-300 เริ่มที่ 5 มิลลิกรัม วันละ 1 ครั้ง ตอนเช้า ปรับเพิ่มได้ทุก 4 สัปดาห์ ขนาดสูงสุด 10 มิลลิกรัมต่อวัน ผลข้างเคียงที่พบบ่อยคือข้อเท้าบวม

## 4. ข้อห้ามใช้
ห้ามใช้เพรสซิลอลในหญิงตั้งครรภ์และหญิงที่วางแผนตั้งครรภ์ ให้เปลี่ยนเป็นเมโทดีปา (Methodipa) รหัสยา MTD-250 และส่งปรึกษาสูติแพทย์ภายใน 1 สัปดาห์

## 5. เป้าหมาย
เป้าหมายความดันต่ำกว่า 130/80 มิลลิเมตรปรอทในผู้ป่วยเบาหวานหรือโรคไตเรื้อรัง และต่ำกว่า 140/90 มิลลิเมตรปรอทในผู้ป่วยทั่วไป นัดติดตามทุก 4 สัปดาห์จนถึงเป้าหมาย แล้วนัดทุก 3 เดือน
"""

DOCS["PROTOCOL-SEPSIS-2567.md"] = """# แนวปฏิบัติภาวะติดเชื้อในกระแสเลือด (sepsis) โรงพยาบาลสมมติเวช ฉบับ พ.ศ. 2567

> เอกสารสมมติเพื่อการสอนเท่านั้น โรงพยาบาล ชื่อยา และเกณฑ์ทั้งหมดแต่งขึ้น ห้ามใช้อ้างอิงทางคลินิก

รหัสเอกสาร: PROTOCOL-SEPSIS-2567
สถานะ: ใช้งานปัจจุบัน

## 1. การคัดกรอง
สงสัย sepsis เมื่อผู้ป่วยมีแหล่งติดเชื้อที่สงสัย ร่วมกับอย่างน้อย 2 ข้อต่อไปนี้ คือ อัตราการหายใจตั้งแต่ 22 ครั้งต่อนาที ความดันซิสโตลิกไม่เกิน 100 มิลลิเมตรปรอท หรือระดับความรู้สึกตัวเปลี่ยนแปลง

## 2. สิ่งที่ต้องทำภายใน 1 ชั่วโมงแรก
1. เจาะเลือดเพาะเชื้อ 2 ขวดจากคนละตำแหน่ง ก่อนให้ยาปฏิชีวนะ
2. ตรวจระดับแลกเตตในเลือด
3. ให้ยาปฏิชีวนะแอนติแบค-ซี (Antibac-C) รหัสยา ABC-500 ขนาด 2 กรัม ทางหลอดเลือดดำ ทุก 8 ชั่วโมง
4. ให้สารน้ำ 30 มิลลิลิตรต่อกิโลกรัม เมื่อความดันต่ำหรือแลกเตตตั้งแต่ 4 มิลลิโมลต่อลิตร

## 3. ผู้ป่วยที่แพ้ยากลุ่มเพนิซิลลิน
ตรวจประวัติแพ้ยาก่อนให้แอนติแบค-ซีทุกครั้ง ถ้าผู้ป่วยมีประวัติแพ้ยากลุ่มเพนิซิลลิน ห้ามให้แอนติแบค-ซี ให้ใช้เมโรแซน (Merozan) รหัสยา MRZ-100 ขนาด 1 กรัม ทางหลอดเลือดดำ ทุก 8 ชั่วโมงแทน

## 4. การปรับขนาดยาตามการทำงานของไต
ผู้ป่วยที่ eGFR ต่ำกว่า 30 ให้ลดแอนติแบค-ซีเป็น 1 กรัม ทุก 12 ชั่วโมง หลังให้ขนาดแรกเต็มขนาดแล้ว

## 5. การประเมินซ้ำ
ประเมินสัญญาณชีพทุก 15 นาทีในชั่วโมงแรก ตรวจแลกเตตซ้ำภายใน 2 ถึง 4 ชั่วโมงถ้าค่าแรกสูงกว่า 2 มิลลิโมลต่อลิตร และทบทวนยาปฏิชีวนะเมื่อได้ผลเพาะเชื้อ
"""

DOCS["FORMULARY-2568.md"] = """# บัญชียาโรงพยาบาลสมมติเวช ฉบับ พ.ศ. 2568

> เอกสารสมมติเพื่อการสอนเท่านั้น โรงพยาบาล ชื่อยา และเกณฑ์ทั้งหมดแต่งขึ้น ห้ามใช้อ้างอิงทางคลินิก

รหัสเอกสาร: FORMULARY-2568
สถานะ: ใช้งานปัจจุบัน

## รายการยาที่อ้างถึงในแนวปฏิบัติ
| รหัสยา | ชื่อยา | รูปแบบ | กลุ่มยา | สถานะในบัญชี |
|---|---|---|---|---|
| GLN-850 | กลูโคนอร์ม (Gluconorm) | เม็ด 850 มก. | ยาลดน้ำตาลในเลือด | ใช้ได้ |
| DBX-100 | ไดอะเบกซ์ (Diabex-R) | เม็ด 100 มก. | ยาลดน้ำตาลในเลือด | ใช้ได้ |
| SGD-120 | ซูการ์ดิน (Sugardin) | เม็ด 120 มก. | ยาลดน้ำตาลในเลือด | จำกัดการใช้ ดูประกาศ MEMO-2568-014 |
| PSL-300 | เพรสซิลอล (Pressilol) | เม็ด 5 มก. | ยาลดความดันโลหิต | ใช้ได้ |
| MTD-250 | เมโทดีปา (Methodipa) | เม็ด 250 มก. | ยาลดความดันโลหิต | ใช้ได้ |
| ABC-500 | แอนติแบค-ซี (Antibac-C) | ผงสำหรับฉีด 1 ก. | ยาปฏิชีวนะ | ใช้ได้ |
| MRZ-100 | เมโรแซน (Merozan) | ผงสำหรับฉีด 1 ก. | ยาปฏิชีวนะ | ต้องมีใบสั่งจากแพทย์เฉพาะทาง |

## หมายเหตุการเก็บรักษา
กลูโคนอร์มและเพรสซิลอลเก็บที่อุณหภูมิห้อง ไม่เกิน 30 องศาเซลเซียส แอนติแบค-ซีและเมโรแซนหลังผสมแล้วต้องใช้ภายใน 6 ชั่วโมง

## การเบิกยานอกเวลาราชการ
ยาปฏิชีวนะสำหรับ sepsis เบิกได้ตลอด 24 ชั่วโมงที่ห้องยาฉุกเฉิน ชั้น 1 อาคารอุบัติเหตุ
"""

DOCS["MEMO-2568-014.md"] = """# ประกาศคณะกรรมการเภสัชกรรมและการบำบัด โรงพยาบาลสมมติเวช ที่ 014/2568

> เอกสารสมมติเพื่อการสอนเท่านั้น โรงพยาบาล ชื่อยา และเกณฑ์ทั้งหมดแต่งขึ้น ห้ามใช้อ้างอิงทางคลินิก

รหัสเอกสาร: MEMO-2568-014
เรื่อง: จำกัดการใช้ยาซูการ์ดิน (Sugardin) รหัสยา SGD-120
วันที่ประกาศ: 15 มีนาคม 2568

## 1. เหตุผล
จากการทบทวนรายงานอาการไม่พึงประสงค์ภายในโรงพยาบาล พบภาวะน้ำตาลต่ำรุนแรงในผู้ป่วยสูงอายุที่ใช้ยาซูการ์ดินบ่อยกว่ายาทางเลือก คณะกรรมการจึงมีมติจำกัดการใช้

## 2. ข้อกำหนด
1. ห้ามเริ่มยาซูการ์ดินในผู้ป่วยรายใหม่ทุกราย ตั้งแต่วันที่ 1 เมษายน 2568
2. ผู้ป่วยเดิมที่ใช้ยาอยู่ ให้แพทย์พิจารณาเปลี่ยนเป็นยาไดอะเบกซ์ (Diabex-R) รหัสยา DBX-100 ในการนัดครั้งถัดไป และต้องเปลี่ยนให้ครบทุกรายภายในวันที่ 30 กันยายน 2568
3. ผู้ป่วยอายุตั้งแต่ 70 ปีขึ้นไปที่ยังใช้ยาซูการ์ดิน ให้โทรนัดมาพบแพทย์ภายใน 4 สัปดาห์ ไม่ต้องรอนัดเดิม

## 3. การเปลี่ยนยา
หยุดซูการ์ดินแล้วเริ่มไดอะเบกซ์ 100 มิลลิกรัม วันละ 1 ครั้ง ในวันถัดไป ไม่ต้องเว้นช่วง และนัดตรวจน้ำตาลปลายนิ้วภายใน 2 สัปดาห์

## 4. ผู้ประสานงาน
กลุ่มงานเภสัชกรรม โทรภายใน 1420 ในเวลาราชการ
"""

for name, text in DOCS.items():
    with open("rag-med-dataset/guidelines/" + name, "w", encoding="utf-8") as f:
        f.write(text)
print(len(DOCS), "ไฟล์:", sorted(DOCS))

### ถามโมเดลโดยไม่มี RAG

ยาในชุดข้อมูลเป็นยาสมมติ โมเดลไม่มีทางรู้จัก สังเกตว่าโมเดลตอบอย่างไร: ตอบมั่นใจ แต่งเอง หรือบอกว่าไม่รู้ **เก็บคำตอบนี้ไว้เทียบในขั้น 4**

In [ ]:
import ollama

def ask(prompt):
    r = ollama.chat(model="qwen3:8b",
        messages=[{"role": "user", "content": prompt}],
        think=False, options={"temperature": 0})
    return r["message"]["content"]

q = "ผู้ป่วยที่ eGFR 35 ควรปรับขนาดยากลูโคนอร์ม (Gluconorm) อย่างไร"
answer_no_rag = ask(q)
print(answer_no_rag)

## ขั้น 1 · Embedding และ similarity

`embed()` แปลงข้อความเป็นเวกเตอร์ แล้ว normalize ให้ยาว 1 ทำให้ dot product เท่ากับ cosine similarity

In [ ]:
import numpy as np

def embed(texts):
    r = ollama.embed(model="bge-m3", input=texts)
    v = np.array(r["embeddings"], dtype="float32")
    return v / np.linalg.norm(v, axis=1, keepdims=True)

sents = ["ผู้ป่วยไตเสื่อมต้องลดขนาดยา",
         "ปรับยาเมื่อ eGFR ต่ำ",
         "วิธีวัดความดันโลหิตที่บ้าน",
         "เกณฑ์วินิจฉัยความดันโลหิตสูง",
         "ที่จอดรถของโรงพยาบาล"]
E = embed(sents)
print("ขนาดเวกเตอร์:", E.shape)
print(np.round(E @ E.T, 2))

**สังเกต:** ประโยค 1 กับ 2 ใช้คำต่างกันแต่ความหมายใกล้กัน คะแนนควรสูงกว่าคู่ที่ไม่เกี่ยวกัน เช่น ประโยค 1 กับ 5

## ขั้น 2 · โหลดเอกสาร ตัด chunk สร้าง index

In [ ]:
import glob, os
docs = []
for p in sorted(glob.glob("rag-med-dataset/guidelines/*.md")):
    docs.append({"doc_id": os.path.basename(p),
                 "text": open(p, encoding="utf-8").read()})

CHUNK_SIZE, OVERLAP = 400, 60
chunks = []
for d in docs:
    t = d["text"]
    for s in range(0, len(t), CHUNK_SIZE - OVERLAP):
        chunks.append({"doc_id": d["doc_id"],
                       "text": t[s:s + CHUNK_SIZE]})
print(len(docs), "ไฟล์ ->", len(chunks), "chunks")

In [ ]:
# ดูตัวอย่าง chunk: สังเกตว่าตัดกลางประโยคหรือกลางตารางหรือไม่
print(chunks[5]["doc_id"])
print(chunks[5]["text"])

In [ ]:
import faiss
X = embed([c["text"] for c in chunks])
index = faiss.IndexFlatIP(X.shape[1])     # IP = inner product = cosine เพราะ normalize แล้ว
index.add(X)
print(index.ntotal, "เวกเตอร์ ขนาด", X.shape[1])

## ขั้น 3 · Retrieve

แปลงคำถามเป็นเวกเตอร์ด้วยโมเดลเดียวกัน แล้วหา k chunk ที่คะแนนสูงสุด

In [ ]:
def retrieve(q, k=3):
    scores, ids = index.search(embed([q]), k)
    return [(chunks[i], float(s))
            for i, s in zip(ids[0], scores[0])]

for c, s in retrieve(q):
    print(round(s, 3), c["doc_id"])
    print(c["text"][:120], "...")
    print()

**ตรวจ:** เอกสารถูกฉบับไหม · chunk มีคำตอบครบไหม · คะแนนอันดับ 1 ห่างจากอันดับ 3 แค่ไหน

## ขั้น 4 · Generate

ประกอบ prompt 3 ส่วน: คำสั่ง, CONTEXT, คำถาม แล้วส่งให้ `ask()` ตัวเดิม

In [ ]:
def rag(q, k=3):
    hits = retrieve(q, k)
    context = "\n\n".join(
        f"[{c['doc_id']}]\n{c['text']}" for c, _ in hits)
    prompt = f"""ตอบคำถามโดยใช้ข้อมูลใน CONTEXT เท่านั้น
ถ้าไม่พบคำตอบใน CONTEXT ให้ตอบว่า "ไม่พบข้อมูลในเอกสาร"
ระบุชื่อเอกสารที่ใช้อ้างอิงท้ายคำตอบ

CONTEXT:
{context}

คำถาม: {q}"""
    return ask(prompt), hits

answer, hits = rag(q)
print(answer)

In [ ]:
print("=== ไม่มี RAG (ขั้น 0) ===")
print(answer_no_rag)
print()
print("=== มี RAG ===")
print(answer)
print()
print("chunk ที่ใช้:", [c["doc_id"] for c, _ in hits])

## ขั้น 5 · ทดลองพัง

### 5.1 คำถามที่ไม่มีในเอกสาร
การค้นคืน k chunk เสมอ แม้ไม่มีอะไรเกี่ยวข้อง การปฏิเสธจึงขึ้นกับ prompt และโมเดล

In [ ]:
q2 = "โรงพยาบาลเปิดรับบริจาคเลือดวันไหน"
answer, hits = rag(q2)
print(answer)
print()
for c, s in hits:
    print(round(s, 3), c["doc_id"])

### 5.2 เปลี่ยน k

In [ ]:
for k in [1, 3, 8]:
    a, _ = rag(q, k)
    print(f"--- k = {k} ---")
    print(a)
    print()

### 5.3 เปลี่ยน CHUNK_SIZE
ฟังก์ชันนี้ตัด chunk และสร้าง index ใหม่ ลอง 200 และ 1000 แล้วถามคำถามเดิม เสร็จแล้วเรียก `rebuild(400, 60)` เพื่อกลับค่าเดิม

In [ ]:
def rebuild(chunk_size, overlap):
    global chunks, index
    chunks = []
    for d in docs:
        t = d["text"]
        for s in range(0, len(t), chunk_size - overlap):
            chunks.append({"doc_id": d["doc_id"],
                           "text": t[s:s + chunk_size]})
    X = embed([c["text"] for c in chunks])
    index = faiss.IndexFlatIP(X.shape[1])
    index.add(X)
    print("chunk_size", chunk_size, "->", len(chunks), "chunks")

for size, ov in [(200, 30), (1000, 150)]:
    rebuild(size, ov)
    a, _ = rag(q)
    print(a)
    print()

rebuild(400, 60)

### 5.4 ลบคำสั่ง "ใช้ CONTEXT เท่านั้น"
ส่ง context ให้เหมือนเดิม แต่ไม่บังคับให้ตอบจากเอกสาร แล้วถามคำถามที่ไม่มีในเอกสาร

In [ ]:
def rag_loose(q, k=3):
    hits = retrieve(q, k)
    context = "\n\n".join(
        f"[{c['doc_id']}]\n{c['text']}" for c, _ in hits)
    prompt = f"""CONTEXT:
{context}

คำถาม: {q}"""
    return ask(prompt), hits

a, _ = rag_loose(q2)
print(a)

## สรุป: โค้ดกับ pipeline

| ขั้นใน pipeline | โค้ดที่เขียนวันนี้ |
|---|---|
| Load, Chunk | loop ตัด chunk |
| Embed | `embed()` |
| Store | `faiss.IndexFlatIP` |
| Retrieve | `retrieve()` |
| Augment, Generate | `rag()` และ `ask()` |